# 3_7 — Publication figures & headline table (P2)

Built on cross-validation artifacts (3_4/3_5); no torch needed. All figures are in English with **object-cluster bootstrap 95% CIs** (not naive fold-CI). Saved to `results/analysis_figs/`.
Figures: (1) reliability diagram, (2) forest plots (AUPRC, post-prefix RMSE) with cluster CIs, (3) onset-vs-trajectory Pareto trade-off, (4) headline table.

In [1]:
import os, sys, json
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB=os.path.join(REPO,'results','tables'); os.makedirs(PUB, exist_ok=True)
FIGS=os.path.join(REPO,'results','analysis_figs'); os.makedirs(FIGS, exist_ok=True)

In [2]:
from liquefaction_ai.evaluation import reliability_diagram, forest_plot, pareto_plot, headline_table
from IPython.display import display
REF = 'DPI-Flow'
samples = pd.read_csv(os.path.join(TABLES,'cv_grouped_samples.csv'))
summary = pd.read_csv(os.path.join(TABLES,'cv_grouped_summary.csv'))
clpath = os.path.join(TABLES,'significance_grouped_cluster_bootstrap.csv')
cluster = pd.read_csv(clpath) if os.path.exists(clpath) else None

## 1. Calibration — reliability diagram

In [3]:
fig, ece = reliability_diagram(samples, ref=REF, bins=10, out_dir=FIGS, suffix='grouped')
print('ECE =', round(ece, 4)); display(fig)

ECE = 0.0082


/Users/nikita/Desktop/projects/liquefaction-ai/src/liquefaction_ai/evaluation/publication.py:86: UserWarning: Glyph 8733 (\N{PROPORTIONAL TO}) missing from font(s) Lucida Grande.
  fig.tight_layout()
/Users/nikita/Desktop/projects/liquefaction-ai/src/liquefaction_ai/evaluation/publication.py:90: UserWarning: Glyph 8733 (\N{PROPORTIONAL TO}) missing from font(s) Lucida Grande.
  fig.savefig(os.path.join(out_dir, f"reliability_{ref}_{suffix}.{ext}"), dpi=150)


<Figure size 460x440 with 1 Axes>

## 2. Model comparison with cluster-bootstrap CIs (forest plots)

In [4]:
if cluster is not None:
    display(forest_plot(cluster, metric='AUPRC', higher_better=True, ref=REF, out_dir=FIGS))
    if 'traj_rmse_continuation' in cluster.columns:
        display(forest_plot(cluster, metric='traj_rmse_continuation', higher_better=False, ref=REF,
                            title='Post-prefix trajectory RMSE', out_dir=FIGS))
else:
    print('run 3_5 first to produce significance_grouped_cluster_bootstrap.csv')

<Figure size 640x590 with 1 Axes>

<Figure size 640x540 with 1 Axes>

## 3. Onset vs trajectory trade-off (Pareto balance)

In [5]:
display(pareto_plot(summary, x='Traj_RMSE_continuation', y='AUPRC', ref=REF, out_dir=FIGS))

<Figure size 620x500 with 1 Axes>

## 4. Publication headline table (CI from cluster bootstrap)

In [6]:
hl = headline_table(summary, cluster_df=cluster)
hl.to_csv(os.path.join(PUB,'publication_headline_grouped.csv'), index=False)
hl

,model,AUPRC ↑ (primary onset),Brier ↓,ECE ↓ (calibration),Coverage@90 → 0.90,PPR CRPS ↓ (proper),PPR RMSE ↓ (post-prefix),PPR RMSE worst-state ↓,N_liq log-MAE ↓ (censored-aware),N_liq log-MAE ↓ (liquefied-only),CRR RMSE ↓ (DPI family),Physics violations ↓,AUROC ↑ (reference only)
0,EVT-NeuralSSM,"0.960 [0.907, 0.998]","0.067 [0.018, 0.125]","0.058 [0.020, 0.121]","0.818 [0.755, 0.881]",0.062,"0.104 [0.069, 0.137]",0.162,"0.263 [0.149, 0.381]",0.306,0.203,0.000,"0.958 [0.900, 0.997]"
1,DPI-Flow,"0.978 [0.947, 0.999]","0.054 [0.015, 0.103]","0.050 [0.015, 0.102]","0.839 [0.775, 0.897]",0.042,"0.082 [0.056, 0.109]",0.112,"0.248 [0.169, 0.328]",0.352,0.207,0.002,"0.978 [0.952, 0.998]"
2,DPI-EVT,"0.972 [0.930, 0.997]","0.077 [0.027, 0.137]","0.079 [0.026, 0.144]","0.794 [0.677, 0.884]",0.049,"0.111 [0.076, 0.149]",0.127,"0.250 [0.192, 0.314]",0.311,0.159,0.000,"0.971 [0.943, 0.995]"
3,Transformer,"0.922 [0.866, 0.999]","0.061 [0.014, 0.134]","0.052 [0.019, 0.135]","0.939 [0.886, 0.980]",0.026,"0.033 [0.022, 0.047]",0.096,"0.546 [0.416, 0.676]",0.672,—,0.089,"0.926 [0.821, 0.998]"
4,Neural Spline Flow,"0.980 [0.943, 0.998]","0.054 [0.024, 0.090]","0.026 [0.021, 0.079]","0.726 [0.668, 0.779]",0.061,"0.082 [0.060, 0.102]",0.130,"0.738 [0.593, 0.877]",0.878,—,0.799,"0.977 [0.954, 0.996]"
5,GRU,"0.859 [0.652, 0.975]","0.142 [0.085, 0.204]","0.092 [0.062, 0.206]","0.868 [0.774, 0.942]",0.067,"0.105 [0.064, 0.149]",0.187,"1.615 [1.169, 2.167]",1.470,—,0.514,"0.859 [0.737, 0.956]"
6,PINN,"0.871 [0.786, 0.993]","0.107 [0.048, 0.182]","0.074 [0.036, 0.167]","0.768 [0.682, 0.848]",0.085,"0.099 [0.069, 0.130]",0.240,"0.934 [0.654, 1.218]",1.480,—,0.000,"0.889 [0.758, 0.987]"
7,TCN,"0.862 [0.758, 0.982]","0.126 [0.067, 0.194]","0.098 [0.053, 0.193]","0.789 [0.699, 0.877]",0.084,"0.100 [0.067, 0.137]",0.266,"1.076 [0.817, 1.367]",1.745,—,0.677,"0.874 [0.752, 0.966]"
8,CatBoost,"0.996 [0.987, 1.000]","0.033 [0.004, 0.072]","0.030 [0.005, 0.077]",—,—,—,—,"0.215 [0.171, 0.263]",0.245,—,—,"0.994 [0.983, 0.999]"


In [7]:
# === A/B flow vs gaussian: разница (gaussian − flow) с object-bootstrap 95% CI (#3) ===
import os, pandas as pd
_ab_csv = os.path.join(TABLES, "ab_flow_vs_gaussian_pooled.csv")
if os.path.exists(_ab_csv):
    from liquefaction_ai.viz import new_figure, save_figure
    ab = pd.read_csv(_ab_csv); d = ab.dropna(subset=["diff_gauss_minus_flow"]).reset_index(drop=True)
    figw, fig = new_figure((7.0, 2.6)); ax = fig.add_subplot(111); yy = list(range(len(d)))
    ax.errorbar(d["diff_gauss_minus_flow"], yy,
                xerr=[d["diff_gauss_minus_flow"]-d["ci95_low"], d["ci95_high"]-d["diff_gauss_minus_flow"]],
                fmt="o", color="#0b6efd", capsize=4)
    ax.axvline(0.0, ls="--", color="#c46b6b", lw=1.2)
    ax.set_yticks(yy); ax.set_yticklabels(d["metric"])
    ax.set_xlabel("gaussian − flow  (>0 → flow better; lower NLL/CRPS = better)")
    ax.set_title("A/B: conditional flow vs Gaussian posterior (pooled OOF site-bootstrap 95% CI)")
    save_figure(figw, "3_7_ab_flow_vs_gaussian", True); display(ab.round(4))
else:
    print("Нет ab_flow_vs_gaussian_pooled.csv — сначала прогони pooled A/B в 3_4.")


Нет ab_flow_vs_gaussian.csv — сначала прогони A/B-ячейку в 3_1 (RUN_AB=True).


## Calibration headline — empirical site-held-out conformal coverage
Честная калибровка под site-shift: полоса калибруется на VAL-объектах, покрытие меряется на TEST-объектах (модель их не видела), агрегировано по фолдам + object-bootstrap CI. Это, а не gaussian Coverage@90 на single-split, — публикуемое число калибровки.

In [ ]:
import os, pandas as pd
_cf = os.path.join(TABLES, 'cv_object_conformal.csv')
if os.path.exists(_cf):
    conf = pd.read_csv(_cf)
    display(conf[[c for c in ['model','Coverage_emp','Coverage_lo','Coverage_hi','mean_band_width','n_objects'] if c in conf.columns]].round(3))
else:
    print('run 3_4 first to produce cv_object_conformal.csv (empirical site-held-out coverage)')

# Финальный manifest пишется ПОСЛЕ таблиц и включает SHA256 result CSV/JSON и weights.
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation import write_run_manifest, validate_run_manifest
_pop, _cfg = load_population_artifact(os.path.join(REPO, 'data', 'dataset'))
_manifest = write_run_manifest(_pop, _cfg, REPO, 'models', require_clean=True)
_manifest_data = json.load(open(_manifest))
validate_run_manifest(_manifest_data, required_models=('dpi_flow','dpi_evt','evt_ssm','pinn','transformer','nsf','gru','tcn','catboost'), required_results=('cv_grouped_raw.csv','cv_grouped_samples.csv','significance_grouped_cluster_bootstrap.csv','ab_flow_vs_gaussian_pooled.csv','publication_headline_grouped.csv'))
print('final manifest validated:', _manifest)